- 读取文档
- embeddings
- vectorstores
- failure modes

读取文档

In [1]:
import numpy as np
from langchain_community.document_loaders import PyPDFLoader

# 加载pdf
loaders = [
    PyPDFLoader("data/matplotlib/第一回：Matplotlib初相识 — fantastic-matplotlib.pdf"),
    PyPDFLoader("data/matplotlib/第二回：艺术画笔见乾坤 — fantastic-matplotlib.pdf"),
    PyPDFLoader("data/matplotlib/第三回：布局格式定方圆 — fantastic-matplotlib.pdf"),
    PyPDFLoader("data/matplotlib/第四回：文字图例尽眉目 — fantastic-matplotlib.pdf"),
    PyPDFLoader("data/matplotlib/第五回：样式色彩秀芳华 — fantastic-matplotlib.pdf")
]

docs = []
for loader in loaders:
    docs.extend(loader.load())

In [2]:
# 分割文本
from langchain_text_splitters import RecursiveCharacterTextSplitter

r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=150
)

splits = r_splitter.split_documents(docs)
print(len(splits))

49


embeddings

In [3]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
embedding = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

sentence1 = "i like dogs"
sentence2 = "i like canines"
sentence3 = "the weather is ugly outside"

embedding1 = embedding.embed_query(sentence1)
embedding2 = embedding.embed_query(sentence2)
embedding3 = embedding.embed_query(sentence3)

# 通过点积展示嵌入情况
print(f"第一句和第二句：{np.dot(embedding1, embedding2)}")
print(f"第一句和第三句：{np.dot(embedding1, embedding3)}")
print(f"第二句和第三句：{np.dot(embedding2, embedding3)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


第一句和第二句：0.9151645891805045
第一句和第三句：0.08337089578162207
第二句和第三句：0.040403691508289


In [4]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
embedding_chinese = HuggingFaceEmbeddings(
    model_name="shibing624/text2vec-base-chinese",
    model_kwargs={"device": "cpu"}
)

sentence1_chinese = "我喜欢狗"
sentence2_chinese = "我喜欢犬科动物"
sentence3_chinese = "今天外面天气真糟糕"

embedding1_chinese = embedding_chinese.embed_query(sentence1_chinese)
embedding2_chinese = embedding_chinese.embed_query(sentence2_chinese)
embedding3_chinese = embedding_chinese.embed_query(sentence3_chinese)

model.safetensors:   0%|          | 0.00/409M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: shibing624/text2vec-base-chinese
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/319 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

In [5]:
# 通过点积展示嵌入情况
print(f"第一句和第二句：{np.dot(embedding1_chinese, embedding2_chinese)}")
print(f"第一句和第三句：{np.dot(embedding1_chinese, embedding3_chinese)}")
print(f"第二句和第三句：{np.dot(embedding2_chinese, embedding3_chinese)}")

第一句和第二句：296.3256599942579
第一句和第三句：77.78273062726015
第二句和第三句：78.8383627087751


vectorstores

In [6]:
### 初始化向量存储库 chroma
from langchain_community.vectorstores import Chroma

# 定义持久化路径，使数据库可以保存在本地磁盘上，而不是只在内存中运行
persist_directory = 'data/matplotlib/'

# 从已加载的文档中创建一个向量数据库
vectordb = Chroma.from_documents(
    documents=splits,
    embedding=embedding_chinese,
    persist_directory=persist_directory
)

In [7]:
print(vectordb._collection.count())

49


In [13]:
### Similarity Search
question = 'matplotlib 是什么'

answers = vectordb.similarity_search_with_score(question, k=3)

print(answers[0])

(Document(metadata={'moddate': '2026-03-17T09:45:57+00:00', 'total_pages': 15, 'source': 'data/matplotlib/第二回：艺术画笔见乾坤 — fantastic-matplotlib.pdf', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36', 'title': '第二回：艺术画笔见乾坤 — fantastic-matplotlib', 'page_label': '1', 'creationdate': '2026-03-17T09:45:57+00:00', 'producer': 'Skia/PDF m145', 'page': 0}, page_content='第⼆回：艺术画笔⻅乾坤\nimport numpy as np\nimport pandas as pd\nimport re\nimport matplotlib\nimport matplotlib.pyplot as plt\nfrom matplotlib.lines import Line2D   \nfrom matplotlib.patches import Circle, Wedge\nfrom matplotlib.collections import PatchCollection\n⼀、概述\n1. matplotlib 的三层 api\nmatplotlib 的原理或者说基础逻辑是，⽤ Artist 对象在画布 (canvas) 上绘制 (Render) 图形。\n就和⼈作画的步骤类似：\n1. 准备⼀块画布或画纸\n2. 准备好颜料、画笔等制图⼯具\n3. 作画\n所以 matplotlib 有三个层次的 API ：\nmatplotlib.backend_bases.FigureCanvas 代表了绘图区，所有的图像都是在绘图区完成的\nmatplotlib.backend_bases.Renderer 代表了渲染器，可以近似理解为画笔，控制如何在  FigureCanvas 

failure modes:
- 重复块：搜索到的所有相似文档并没有强制多样性；
- 检索错误答案